# EMT SSN synchronous generator grid-connected example

Python/Jupyter translation of `EMT_SSN_SynchronousGenerator_Test.cpp`.

The control parameters and synchron generator model are derived from "C. Dufour, "Highly Stable Rotating Machine Models Using the State-Space-Nodal Real-Time Solver," 2018 IEEE Workshop on Complexity in Engineering (COMPENG), Florence, Italy, 2018, pp. 1-10, doi: 10.1109/CompEng.2018.8536236."
https://ieeexplore.ieee.org/abstract/document/8536236


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import dpsimpy

emt = dpsimpy.emt
sp = dpsimpy.sp

ph3 = emt.ph3
sp_ph1 = sp.ph1

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
PhaseType = dpsimpy.PhaseType
PowerflowBusType = dpsimpy.PowerflowBusType
Solver = dpsimpy.Solver
SolverBehaviour = dpsimpy.SolverBehaviour
SwitchEvent3Ph = dpsimpy.event.SwitchEvent3Ph

## Simulation configuration and machine ratings

In [ ]:
final_time = 3.0
time_step = 100e-6

sim_name = "EMT_SSN_SynchronousGenerator_GridConnected"
sim_name_pf = sim_name + "_PF"
sim_name_emt = sim_name + "_EMT"

nominal_apparent_power = 202e6
nominal_voltage = 13.8e3
nominal_frequency = 60.0
nominal_omega = 2.0 * np.pi * nominal_frequency

pole_pairs = 1
nominal_mechanical_speed = nominal_omega / pole_pairs

generated_active_power_pu = 0.8
generated_reactive_power_pu = 0.0

generated_active_power = generated_active_power_pu * nominal_apparent_power
generated_reactive_power = generated_reactive_power_pu * nominal_apparent_power

## Base quantities and per-unit machine parameters

In [ ]:
impedance_base = nominal_voltage**2 / nominal_apparent_power
inductance_base = impedance_base / nominal_omega
current_base = nominal_apparent_power / nominal_voltage
torque_base = nominal_apparent_power / nominal_mechanical_speed

stator_resistance_pu = 3.08e-3
stator_leakage_inductance_pu = 0.124

mutual_inductance_d_pu = 1.29
mutual_inductance_q_pu = 0.388

field_resistance_pu = 9.56e-4
field_leakage_inductance_pu = 0.1228

damper_resistance_d_pu = 1.262e-2
damper_leakage_inductance_d_pu = 1.96e-1

damper_resistance_q1_pu = 2.12e-2
damper_leakage_inductance_q1_pu = 1.44e-1

inertia_constant = 3.0

## Convert the machine parameters to SI units

In [ ]:
stator_resistance = stator_resistance_pu * impedance_base
field_resistance = field_resistance_pu * impedance_base
damper_resistance_d = damper_resistance_d_pu * impedance_base
damper_resistance_q1 = damper_resistance_q1_pu * impedance_base

mutual_inductance_d = mutual_inductance_d_pu * inductance_base
mutual_inductance_q = mutual_inductance_q_pu * inductance_base

stator_inductance_d = (
    stator_leakage_inductance_pu + mutual_inductance_d_pu
) * inductance_base

stator_inductance_q = (
    stator_leakage_inductance_pu + mutual_inductance_q_pu
) * inductance_base

field_inductance = (
    field_leakage_inductance_pu + mutual_inductance_d_pu
) * inductance_base

damper_inductance_d = (
    damper_leakage_inductance_d_pu + mutual_inductance_d_pu
) * inductance_base

damper_inductance_q1 = (
    damper_leakage_inductance_q1_pu + mutual_inductance_q_pu
) * inductance_base

# The paper machine has only one q-axis damper winding. The second q-axis
# damper state of the general model is made electrically inactive.
disabled_damper_q2_pu = 1000.0
damper_resistance_q2 = disabled_damper_q2_pu * impedance_base
damper_inductance_q2 = disabled_damper_q2_pu * inductance_base

rotor_inertia = (
    2.0 * inertia_constant * nominal_apparent_power / nominal_mechanical_speed**2
)

mechanical_damping = 0.0

parameter_table = pd.DataFrame(
    {
        "quantity": [
            "Z_base",
            "L_base",
            "I_base",
            "Rs",
            "Rf",
            "Rkd",
            "Rkq1",
            "Ld",
            "Lq",
            "Lmd",
            "Lmq",
            "Lf",
            "Lkd",
            "Lkq1",
            "J",
        ],
        "value": [
            impedance_base,
            inductance_base,
            current_base,
            stator_resistance,
            field_resistance,
            damper_resistance_d,
            damper_resistance_q1,
            stator_inductance_d,
            stator_inductance_q,
            mutual_inductance_d,
            mutual_inductance_q,
            field_inductance,
            damper_inductance_d,
            damper_inductance_q1,
            rotor_inertia,
        ],
    }
)

display(parameter_table)

## Initial operating point and load-step disturbance

In [ ]:
initial_electrical_angle = 1.95868864
field_voltage = -13.86626959

mechanical_torque = generated_active_power / nominal_mechanical_speed

disturbance_power = 0.05 * nominal_apparent_power
disturbance_resistance = nominal_voltage**2 / disturbance_power

load_step_start_time = 1.0
load_step_end_time = 2.0

switch_open_resistance = 1e9
switch_closed_resistance = 1e-6

grid_connection_resistance = 0.01

print(f"Generated active power: {generated_active_power / 1e6:.3f} MW")
print(f"Mechanical torque: {mechanical_torque:.6g} N m")
print(f"Disturbance power: {disturbance_power / 1e6:.3f} MW")
print(f"Disturbance resistance: {disturbance_resistance:.6g} ohm")

## Helper functions

In [ ]:
def three_phase_parameter(value):
    return dpsimpy.Math.single_phase_parameter_to_three_phase(value)


def log_attr(logger, name, obj, attr_name):
    if hasattr(obj, "attr"):
        logger.log_attribute(name, obj.attr(attr_name))
    else:
        logger.log_attribute(name, obj.attribute(attr_name))


def load_csv(name):
    expected_path = Path("logs") / name / f"{name}.csv"

    if expected_path.exists():
        return pd.read_csv(expected_path, skipinitialspace=True)

    matches = list(Path("logs").glob(f"**/*{name}*.csv"))

    if not matches:
        matches = list((Path("logs") / name).glob("*.csv"))

    if not matches:
        raise FileNotFoundError(f"No CSV output found for simulation '{name}'.")

    print("Using CSV:", matches[0])
    return pd.read_csv(matches[0], skipinitialspace=True)


def signal_columns(df, prefix):
    return [
        column
        for column in df.columns
        if column == prefix or column.startswith(prefix + "_")
    ]


def add_switch_markers(ax):
    ax.axvline(
        load_step_start_time,
        linestyle="--",
        label="Disturbance connected",
    )
    ax.axvline(
        load_step_end_time,
        linestyle="--",
        label="Disturbance disconnected",
    )


def plot_signal_group(df, prefix, ylabel, title):
    columns = signal_columns(df, prefix)

    if not columns:
        print(f"Skipping {prefix}: no matching columns found.")
        return

    time = df.iloc[:, 0].to_numpy()

    fig, ax = plt.subplots(figsize=(12, 4))

    for column in columns:
        ax.plot(time, df[column], label=column)

    add_switch_markers(ax)
    ax.set_xlabel("Time [s]")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True)
    ax.legend()
    plt.show()

# Power-flow initialization

In [ ]:
def build_and_run_powerflow():
    Logger.set_log_dir("logs/" + sim_name_pf)

    n_grid_pf = sp.SimNode("nGrid", PhaseType.Single)
    n_machine_pf = sp.SimNode("nMachine", PhaseType.Single)

    slack_pf = sp_ph1.NetworkInjection("Slack")
    slack_pf.set_parameters(nominal_voltage)
    slack_pf.set_base_voltage(nominal_voltage)
    slack_pf.modify_power_flow_bus_type(PowerflowBusType.VD)

    grid_connection_pf = sp_ph1.PiLine("GridConnection")
    grid_connection_pf.set_parameters(
        grid_connection_resistance,
        0.0,
        0.0,
    )
    grid_connection_pf.set_base_voltage(nominal_voltage)

    # A generator is represented in the power flow by a negative PQ load.
    generator_injection_pf = sp_ph1.Load("SynGen")
    generator_injection_pf.set_parameters(
        -generated_active_power,
        -generated_reactive_power,
        nominal_voltage,
    )
    generator_injection_pf.modify_power_flow_bus_type(PowerflowBusType.PQ)

    slack_pf.connect([n_grid_pf])
    grid_connection_pf.connect([n_grid_pf, n_machine_pf])
    generator_injection_pf.connect([n_machine_pf])

    system_pf = SystemTopology(
        nominal_frequency,
        [n_grid_pf, n_machine_pf],
        [
            slack_pf,
            grid_connection_pf,
            generator_injection_pf,
        ],
    )

    logger_pf = Logger(sim_name_pf)
    log_attr(logger_pf, "Voltage_Grid", n_grid_pf, "v")
    log_attr(logger_pf, "Voltage_Machine", n_machine_pf, "v")

    sim_pf = Simulation(sim_name_pf)
    sim_pf.set_system(system_pf)
    sim_pf.set_time_step(1.0)
    sim_pf.set_final_time(2.0)
    sim_pf.set_domain(Domain.SP)
    sim_pf.set_solver(Solver.NRP)
    sim_pf.set_solver_component_behaviour(SolverBehaviour.Initialization)
    sim_pf.do_init_from_nodes_and_terminals(False)
    sim_pf.add_logger(logger_pf)
    sim_pf.run()

    return system_pf


system_pf = build_and_run_powerflow()

# EMT simulation

In [ ]:
def build_and_run_emt(system_pf):
    Logger.set_log_dir("logs/" + sim_name_emt)

    n_grid = emt.SimNode("nGrid", PhaseType.ABC)
    n_machine = emt.SimNode("nMachine", PhaseType.ABC)
    n_disturbance = emt.SimNode("nDisturbance", PhaseType.ABC)

    slack = ph3.NetworkInjection("Slack")

    generator = ph3.SSN_SynchronousGenerator("SynGen", "SynGen")

    generator.set_parameters(
        nominal_frequency,
        pole_pairs,
        stator_resistance,
        field_resistance,
        damper_resistance_d,
        damper_resistance_q1,
        damper_resistance_q2,
        stator_inductance_d,
        stator_inductance_q,
        mutual_inductance_d,
        mutual_inductance_q,
        field_inductance,
        damper_inductance_d,
        damper_inductance_q1,
        damper_inductance_q2,
        rotor_inertia,
        mechanical_damping,
        field_voltage,
        mechanical_torque,
        initial_electrical_angle,
        True,
    )

    generator.set_numerical_linearization_parameters(
        1e-6,
        1e-8,
    )

    grid_connection = ph3.Resistor("GridConnection")
    grid_connection.set_parameters(three_phase_parameter(grid_connection_resistance))

    load_switch = ph3.Switch("LoadSwitch")
    load_switch.set_parameters(
        three_phase_parameter(switch_open_resistance),
        three_phase_parameter(switch_closed_resistance),
    )
    load_switch.open()

    disturbance_load = ph3.Resistor("DisturbanceLoad")
    disturbance_load.set_parameters(three_phase_parameter(disturbance_resistance))

    slack.connect([n_grid])
    grid_connection.connect([n_grid, n_machine])

    generator.connect([emt.SimNode.gnd, n_machine])

    load_switch.connect([n_machine, n_disturbance])
    disturbance_load.connect([n_disturbance, emt.SimNode.gnd])

    system_emt = SystemTopology(
        nominal_frequency,
        [n_grid, n_machine, n_disturbance],
        [
            slack,
            grid_connection,
            generator,
            load_switch,
            disturbance_load,
        ],
    )

    system_emt.init_with_powerflow(system_pf, Domain.EMT)

    logger_emt = Logger(sim_name_emt)

    log_attr(logger_emt, "Voltage_Grid", n_grid, "v")
    log_attr(logger_emt, "Voltage_Machine", n_machine, "v")
    log_attr(
        logger_emt,
        "Voltage_Disturbance",
        n_disturbance,
        "v",
    )

    log_attr(
        logger_emt,
        "SynGen_InterfaceVoltage",
        generator,
        "v_intf",
    )
    log_attr(
        logger_emt,
        "SynGen_InterfaceCurrent",
        generator,
        "i_intf",
    )
    log_attr(
        logger_emt,
        "SynGen_ElectricalPower",
        generator,
        "electrical_power",
    )
    log_attr(
        logger_emt,
        "SynGen_ElectricalTorque",
        generator,
        "electrical_torque",
    )
    log_attr(
        logger_emt,
        "SynGen_MechanicalSpeed",
        generator,
        "mechanical_speed",
    )
    log_attr(
        logger_emt,
        "SynGen_ElectricalAngle",
        generator,
        "electrical_angle",
    )
    log_attr(
        logger_emt,
        "SynGen_StatorVoltageD",
        generator,
        "stator_voltage_d",
    )
    log_attr(
        logger_emt,
        "SynGen_StatorVoltageQ",
        generator,
        "stator_voltage_q",
    )
    log_attr(
        logger_emt,
        "SynGen_StatorCurrentD",
        generator,
        "stator_current_d",
    )
    log_attr(
        logger_emt,
        "SynGen_StatorCurrentQ",
        generator,
        "stator_current_q",
    )
    log_attr(
        logger_emt,
        "SynGen_FieldCurrent",
        generator,
        "field_current",
    )
    log_attr(
        logger_emt,
        "DisturbanceCurrent",
        disturbance_load,
        "i_intf",
    )

    sim = Simulation(sim_name_emt)

    sim.add_event(
        SwitchEvent3Ph(
            load_step_start_time,
            load_switch,
            True,
        )
    )
    sim.add_event(
        SwitchEvent3Ph(
            load_step_end_time,
            load_switch,
            False,
        )
    )

    sim.set_system(system_emt)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time + time_step)
    sim.set_domain(Domain.EMT)
    sim.set_solver(Solver.MNA)

    sim.do_system_matrix_recomputation(True)

    sim.do_init_from_nodes_and_terminals(True)
    sim.add_logger(logger_emt)
    sim.run()

    return system_emt


system_emt = build_and_run_emt(system_pf)

# Load simulation results

In [ ]:
pf_result = load_csv(sim_name_pf)
emt_result = load_csv(sim_name_emt)

print("Power-flow result:")
display(pf_result.head())

print("EMT result:")
display(emt_result.head())

# Electrical power and torque

In [ ]:
plot_signal_group(
    emt_result,
    "SynGen_ElectricalPower",
    "Electrical power [W]",
    "Synchronous generator electrical power",
)

plot_signal_group(
    emt_result,
    "SynGen_ElectricalTorque",
    "Electrical torque [N m]",
    "Synchronous generator electrical torque",
)

# Mechanical speed and electrical angle

In [ ]:
plot_signal_group(
    emt_result,
    "SynGen_MechanicalSpeed",
    "Mechanical speed [rad/s]",
    "Rotor mechanical speed",
)

plot_signal_group(
    emt_result,
    "SynGen_ElectricalAngle",
    "Electrical angle [rad]",
    "Rotor electrical angle",
)

# dq stator voltages

In [ ]:
time = emt_result.iloc[:, 0].to_numpy()

fig, ax = plt.subplots(figsize=(12, 4))

for column, label in [
    ("SynGen_StatorVoltageD", "v_d"),
    ("SynGen_StatorVoltageQ", "v_q"),
]:
    if column in emt_result.columns:
        ax.plot(time, emt_result[column], label=label)

add_switch_markers(ax)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Voltage [V]")
ax.set_title("Synchronous generator stator voltage in dq coordinates")
ax.grid(True)
ax.legend()
plt.show()

# dq stator currents and field current

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

for column, label in [
    ("SynGen_StatorCurrentD", "i_d"),
    ("SynGen_StatorCurrentQ", "i_q"),
    ("SynGen_FieldCurrent", "i_field"),
]:
    if column in emt_result.columns:
        ax.plot(time, emt_result[column], label=label)

add_switch_markers(ax)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Current [A]")
ax.set_title("Synchronous generator dq and field currents")
ax.grid(True)
ax.legend()
plt.show()

# Three-phase machine-bus voltage

In [ ]:
plot_signal_group(
    emt_result,
    "Voltage_Machine",
    "Voltage [V]",
    "Machine-bus three-phase voltage",
)

# Three-phase generator interface current

In [ ]:
plot_signal_group(
    emt_result,
    "SynGen_InterfaceCurrent",
    "Current [A]",
    "Synchronous generator interface current",
)

# Disturbance current

In [ ]:
plot_signal_group(
    emt_result,
    "DisturbanceCurrent",
    "Current [A]",
    "Switched disturbance current",
)

# Per-unit mechanical speed and electrical power

In [ ]:
speed_columns = signal_columns(
    emt_result,
    "SynGen_MechanicalSpeed",
)
power_columns = signal_columns(
    emt_result,
    "SynGen_ElectricalPower",
)

fig, ax = plt.subplots(figsize=(12, 4))

if speed_columns:
    speed_pu = emt_result[speed_columns[0]].to_numpy() / nominal_mechanical_speed
    ax.plot(time, speed_pu, label="Mechanical speed [pu]")

if power_columns:
    # Positive component electrical power means absorption. Negate it so
    # positive plotted power means generator export.
    generated_power_pu = (
        -emt_result[power_columns[0]].to_numpy() / nominal_apparent_power
    )
    ax.plot(time, generated_power_pu, label="Generated active power [pu]")

ax.axhline(1.0, linestyle=":", label="Nominal speed")
ax.axhline(
    generated_active_power_pu,
    linestyle=":",
    label="Initial generated power",
)
add_switch_markers(ax)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Per-unit value")
ax.set_title("Generator mechanical speed and generated power")
ax.grid(True)
ax.legend()
plt.show()